# Comparar extraccion de texto de un PDF digital - Google Colab

Cuaderno preparado para ejecutarse directamente en Google Colab. Asume que `ejemplo_cmf_digital.pdf` se encuentra cargado en Colab (`/content/ejemplo_cmf_digital.pdf`). Compara resultados por pagina con PyMuPDF, pypdf y pdfplumber, muestra el DataFrame en el cuaderno y crea un Excel en `/content`.

Si subes otro archivo, cambia `PDF_PATH` en la primera celda de codigo. Las librerias extraen la capa de texto y no hacen OCR.


In [ ]:
# Google Colab: instala las dependencias en el entorno de ejecuci?n.
%pip -q install pymupdf pypdf pdfplumber pandas openpyxl

from pathlib import Path
import pandas as pd
import pymupdf
from pypdf import PdfReader
import pdfplumber
from IPython.display import display

PDF_PATH = Path("/content/ejemplo_cmf_digital.pdf")  # PDF cargado en la ra?z de Colab
OUTPUT_XLSX = Path("/content/comparacion_extraccion_pdf.xlsx")

if not PDF_PATH.is_file():
    raise FileNotFoundError(
        f"No se encontr? {PDF_PATH}. Sube ejemplo_cmf_digital.pdf a la ra?z de Colab "
        "(panel Archivos > Subir) o actualiza PDF_PATH con el nombre correcto."
    )
print(f"PDF: {PDF_PATH}")


## Extraer el texto de cada p?gina

La primera p?gina se numera como 1 para facilitar la comparaci?n con el visor PDF. Las p?ginas sin texto extra?ble quedan como cadenas vac?as.


In [ ]:
# PyMuPDF
with pymupdf.open(PDF_PATH) as doc:
    textos_pymupdf = [pagina.get_text() or "" for pagina in doc]

# pypdf
lector = PdfReader(str(PDF_PATH))
textos_pypdf = [pagina.extract_text() or "" for pagina in lector.pages]

# pdfplumber
with pdfplumber.open(PDF_PATH) as pdf:
    textos_pdfplumber = [pagina.extract_text() or "" for pagina in pdf.pages]

longitudes = {len(textos_pymupdf), len(textos_pypdf), len(textos_pdfplumber)}
if len(longitudes) != 1:
    raise ValueError("Las bibliotecas reportaron diferente número de páginas; revisar el PDF.")

resultado = pd.DataFrame({
    "Página": range(1, len(textos_pymupdf) + 1),
    "PyMuPDF": textos_pymupdf,
    "pypdf": textos_pypdf,
    "pdfplumber": textos_pdfplumber,
})
print(f"Páginas comparadas: {len(resultado)}")


## Comparaci?n desplegable

La salida es un DataFrame interactivo de Colab. Cada fila representa una p?gina y cada columna contiene el texto de una librer?a; puedes desplazarte por el contenido de las celdas para comparar textos extensos.


In [ ]:
pd.set_option("display.max_colwidth", None)
display(resultado)


## Exportar a Excel

El Excel se guarda en `/content/comparacion_extraccion_pdf.xlsx`. La hoja incluye una fila por p?gina, una columna por librer?a, ajuste de texto, filtro e inmovilizaci?n de encabezado y numeraci?n. La siguiente celda ofrece descargarlo a tu equipo.


In [ ]:
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    resultado.to_excel(writer, index=False, sheet_name="Textos por pagina")
    hoja = writer.sheets["Textos por pagina"]
    hoja.freeze_panes = "B2"
    hoja.auto_filter.ref = hoja.dimensions
    hoja.column_dimensions["A"].width = 12
    for columna in ("B", "C", "D"):
        hoja.column_dimensions[columna].width = 60
    for fila in hoja.iter_rows(min_row=2, min_col=2, max_col=4):
        for celda in fila:
            celda.alignment = celda.alignment.copy(wrap_text=True, vertical="top")
    for numero_fila in range(2, hoja.max_row + 1):
        hoja.row_dimensions[numero_fila].height = 90

print(f"Excel guardado en: {OUTPUT_XLSX}")


In [ ]:
from google.colab import files
files.download(str(OUTPUT_XLSX))


## Revision

Compara varias paginas con el PDF abierto, en especial el orden de lectura, encabezados, columnas, caracteres especiales y saltos . Diferencias de formato o espacios no implican necesariamente p?rdida de contenido.
